# Retrieving Data from Prometheus
We can retrieve data from Prometheus server via Prometheus http api using prometheus_api_client python package.  To use this package, we must install the following packages:
```
pip install prometheus_api_client
pip install pandas
```

We first import all necessary packages and connect to our local prometheus server.

In [ ]:
from prometheus_api_client import PrometheusConnect,  MetricSnapshotDataFrame, MetricRangeDataFrame
from prometheus_api_client.utils import parse_datetime
from datetime import timedelta

In [ ]:
prom_url = "http://localhost:9090"
prom = PrometheusConnect(url=prom_url, disable_ssl=True)

We can get all metrics available on the server using *all_metrics* function.

In [ ]:
prom.all_metrics()

## Query metrics
We can query a metric using query builder function (get_current_metric_value) or custom query function (custom_query)

In [ ]:
prom.get_current_metric_value(metric_name='node_cpu_seconds_total', label_config={'mode': 'idle'})

In [ ]:
prom.custom_query(query="node_cpu_seconds_total{mode='idle'}[1m]")

We can query data within a specific time interval.  We can also using paging mechanism (or chunk in this case).  In this example, we will fetch the past 6 hours of data for a particular metric in chunks of 1 hour.  Note that the query return *iterable*.

In [ ]:
start_time = parse_datetime("6h")
end_time = parse_datetime("now")
chunk_size = timedelta(hours=1)

In [ ]:
metric_data = prom.get_metric_range_data(
    "node_cpu_seconds_total{mode='idle'}",
    start_time=start_time,
    end_time=end_time,
    chunk_size=chunk_size,
)
for chunk in metric_data:
    print(chunk)

## Dataframe integration
We can use pandas dataframe to perform data analysis and manipulation. The MetricSnapshotDataFrame module converts "current metric value" data to a DataFrame representation, and the MetricRangeDataFrame converts "metric range values" data to a DataFrame representation. 

In [ ]:
r = prom.custom_query(query="node_cpu_seconds_total{mode='idle'}[1m]")
df = MetricSnapshotDataFrame(r)
df.head()

In [ ]:
metric_data = prom.get_metric_range_data(
    "node_cpu_seconds_total{mode='idle'}",  # this is the metric name and label config
    start_time=start_time,
    end_time=end_time,
    chunk_size=chunk_size,
)

In [ ]:
df = MetricRangeDataFrame(metric_data)
df.head(12)